In [194]:
import pandas as pd
import plotly.express as px
from scipy import stats as st

In [195]:
# trip count for every company from 15th to 16th November 2017
company_count_df = pd.read_csv('data/project_sql_result_01.csv')

# average trips per day that ended in every neighbourhood in November
dropoff_count_df = pd.read_csv('data/project_sql_result_04.csv')

# weather conditions and trip duration for trips from Loop to O'Hare Airport on
# Saturdays

weather_df = pd.read_csv('data/project_sql_result_07.csv')

### 1 - Cleaning

#### 1.1 - company_count_df

In [196]:
company_count_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   company_name  64 non-null     str  
 1   trips_amount  64 non-null     int64
dtypes: int64(1), str(1)
memory usage: 1.1 KB


In [197]:
company_count_df['company_name'] = (company_count_df['company_name']
                                    .str.lower()
                                    .str.strip()
)

In [198]:
# check for duplicated company names
company_count_df['company_name'].duplicated().sum()

np.int64(0)

There are 64 company names and there is no null or duplicated values.

#### 1.2 - dropoff_count_df

In [199]:
dropoff_count_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dropoff_location_name  94 non-null     str    
 1   average_trips          94 non-null     float64
dtypes: float64(1), str(1)
memory usage: 1.6 KB


In [200]:
dropoff_count_df['dropoff_location_name'] = (
    dropoff_count_df['dropoff_location_name']
    .str.lower()
    .str.strip()
)

In [201]:
dropoff_count_df['dropoff_location_name'].duplicated().sum()

np.int64(0)

There are 94 dropoff locations and there is no null or duplicated values.

#### 1.3 - weather_df

In [202]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   start_ts            1068 non-null   str    
 1   weather_conditions  1068 non-null   str    
 2   duration_seconds    1068 non-null   float64
dtypes: float64(1), str(2)
memory usage: 25.2 KB


start_df is in string DataType, it should be DateTime.

In [203]:
weather_df.head()

,start_ts,weather_conditions,duration_seconds
0,2017-11-25 16:00:00,Good,2410.0
1,2017-11-25 14:00:00,Good,1920.0
2,2017-11-25 12:00:00,Good,1543.0
3,2017-11-04 10:00:00,Good,2512.0
4,2017-11-11 07:00:00,Good,1440.0


In [204]:
weather_df['start_ts'] = pd.to_datetime(weather_df['start_ts'],
                                        format='%Y-%m-%d %H:%M:%S'
)

In [205]:
weather_df['weather_conditions'] = (weather_df['weather_conditions']
                                    .str.lower()
                                    .str.strip()
)

In [206]:
# Verify low duration trips
weather_df.sort_values('duration_seconds').head()

,start_ts,weather_conditions,duration_seconds
163,2017-11-11 09:00:00,good,0.0
168,2017-11-11 07:00:00,good,0.0
1063,2017-11-25 11:00:00,good,0.0
204,2017-11-18 19:00:00,good,0.0
552,2017-11-04 01:00:00,good,0.0


In [207]:
weather_df[weather_df['duration_seconds'] < 60]

,start_ts,weather_conditions,duration_seconds
163,2017-11-11 09:00:00,good,0.0
168,2017-11-11 07:00:00,good,0.0
204,2017-11-18 19:00:00,good,0.0
552,2017-11-04 01:00:00,good,0.0
801,2017-11-04 09:00:00,good,0.0
1063,2017-11-25 11:00:00,good,0.0


start_df was changed to DateTyme. There is no duplicated rows. There are 6 trips with 0 seconds duration. These can be cancelled trips, but it should be verified with the data acquisition team. As these are few rows, they will be disconsidered.

In [208]:
weather_df = weather_df[weather_df['duration_seconds'] != 0].reset_index(drop=True)

### 2 - Analysis

#### 2.1 - Neighborhoods

In [209]:
# make sure the data is sorted
dropoff_count_df = dropoff_count_df.sort_values('average_trips', ascending=False)

dropoff_count_df.head(10)

,dropoff_location_name,average_trips
0,loop,10727.466667
1,river north,9523.666667
2,streeterville,6664.666667
3,west loop,5163.666667
4,o'hare,2546.900000
5,lake view,2420.966667
6,grant park,2068.533333
7,museum campus,1510.000000
8,gold coast,1364.233333
9,sheffield & depaul,1259.766667


In [210]:
fig = px.bar(dropoff_count_df.head(10),
             x='dropoff_location_name',
             y='average_trips',
             title='Top 10 Destinations',
             labels={'dropoff_location_name':'Neighborhood',
                     'average_trips': 'Average Trips per Day'}
)

fig.show()

Loop is the destination with more average trips per day, 10727.5.

#### 2.2 - Companies

In [211]:
# make sure the data is sorted
company_count_df = company_count_df.sort_values('trips_amount', ascending=False)

company_count_df.head(10)

,company_name,trips_amount
0,flash cab,19558
1,taxi affiliation services,11422
2,medallion leasin,10367
3,yellow cab,9888
4,taxi affiliation service yellow,9299
5,chicago carriage cab corp,9181
6,city service,8448
7,sun taxi,7701
8,star north management llc,7455
9,blue ribbon taxi association inc.,5953


In [212]:
fig = px.bar(company_count_df.head(10),
             x='company_name',
             y='trips_amount',
             title='Top 10 Companies from 15th to 16th November 2017',
             labels={'company_name': 'Company',
                     'trips_amount': 'Trips'}
)

fig.show()

Flash Cab is the company with the highest trip count from 15th to 16th November 2017, 19558.

#### 2.3 - Hypothesis Test

Hypothesis: the average trip duration from Loop to O'Hare International Airport changes on rainy Saturdays.

In [213]:
weather_df['start_ts'].dt.day_name().unique()

<StringArray>
['Saturday']
Length: 1, dtype: str

weather_df is already filtered for trips on Saturday.


In [217]:
fig = px.box(weather_df,
            x='weather_conditions',
            y='duration_seconds',
            points='all',
            title='Distribution of Trip Duration by Weather',
            labels={'duration_seconds': 'duration in seconds',
                    'weather_conditions': 'weather'}
            
)

fig.show()

In [218]:
fig = px.histogram(weather_df,
                   x='duration_seconds',
                   color='weather_conditions',
                   opacity=0.6,
                   barmode='overlay',
                   histnorm='percent',
                   nbins=60,
                   title='Trip Duration Distribution by Weather Condition',
                   labels={'duration_seconds': 'duration in seconds',
                           'weather_conditions': 'weather'}
)

fig.show()



According to the plots, the trips take longer on rainy days.

H0: The average trip duration from Loop to O'Hare International Airport does not change on rainy Saturdays.

H1: The average trip duration changes.

In [216]:
rainy_sample = weather_df[weather_df['weather_conditions'] == 'bad']['duration_seconds']
good_sample = weather_df[weather_df['weather_conditions'] == 'good']['duration_seconds']

results = st.ttest_ind(rainy_sample, good_sample)

alpha = 0.05

print('p-value: ', results.pvalue)

if results.pvalue < alpha:
    print('Null hypothesis rejected')
else:
    print('Null hypothesis cannot be rejected')

p-value:  1.3318772977743222e-11
Null hypothesis rejected


The null hypothesis is rejected, meaning that the average trip duration from Loop to O'Hare International Airport changes on rainy Saturdays. This result, combined with the graphical observation, means that it can be said that trips take longer on rainy days.